In [1]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv('data/EAUI2026_habilidades_socio.csv', dtype={'educ_jh': str, 'ocupacion_jh': str})


/var/folders/nt/0r4npq391dl_zb397t8yhy0c0000gn/T/ipykernel_95051/2107377911.py:1: DtypeWarning: Columns (0: parentesco) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/EAUI2026_habilidades_socio.csv', dtype={'educ_jh': str, 'ocupacion_jh': str})


In [14]:
# ============================================================================
# DERIVACIÓN DE GSE (Índice Socioeconómico AIM-ESOMAR)
# ============================================================================
# Mapea educación y ocupación del jefe/a de hogar a nivel socioeconómico
# Educación (educ_jh): Nuevas etiquetas mapeadas a categorías
# Ocupación (ocupacion_jh): Códigos 1-6 extraídos de la etiqueta
# GSE resultante: AB (alto), C1, C2, C3, D, E (bajo)
 
def _educ_g(e):
    """Mapea etiqueta educacional a categoría."""
    if pd.isna(e):
        return None
    
    e = str(e).lower()
    
    # Mapeo de etiquetas a categorías
    if "sin educación" in e or "básica incompleta" in e:
        return "basica"
    elif "básica completa" in e:
        return "basica"
    elif "media" in e and "incompleta" in e:
        return "media"
    elif "media" in e and "completa" in e:
        return "media"
    elif "técnica" in e or "tecnica" in e:
        return "tecnica"
    elif "universitaria" in e:
        return "universitaria"
    
    return None
 
def _ocup_code(o):
    """Extrae código de ocupación (1-6) de la etiqueta."""
    if pd.isna(o):
        return None
    
    o = str(o).strip()
    # Extraer número al inicio (ej: "3.Obrero..." → 3)
    if o[0].isdigit():
        return int(o[0])
    
    return None
 
# Matriz de cruce educación × ocupación → GSE
_M = {
    (1,"basica"):"E",  (1,"media"):"E",  (1,"tecnica"):"D",  (1,"universitaria"):"D",
    (2,"basica"):"E",  (2,"media"):"D",  (2,"tecnica"):"D",  (2,"universitaria"):"C3",
    (3,"basica"):"D",  (3,"media"):"C3", (3,"tecnica"):"C3", (3,"universitaria"):"C2",
    (4,"basica"):"C3", (4,"media"):"C2", (4,"tecnica"):"C2", (4,"universitaria"):"C1",
    (5,"basica"):"C2", (5,"media"):"C1", (5,"tecnica"):"C1", (5,"universitaria"):"AB",
    (6,"basica"):"C1", (6,"media"):"AB", (6,"tecnica"):"AB", (6,"universitaria"):"AB",
}
 
_ORDEN_GSE = ["AB", "C1", "C2", "C3", "D", "E"]
 
# Aplicar mapeos
educ_cat = df["educ_jh"].apply(_educ_g)
ocup_code = df["ocupacion_jh"].apply(_ocup_code)
 
# Combinar para obtener GSE
gse_values = []
for o, e in zip(ocup_code, educ_cat):
    if pd.isna(o) or e is None:
        gse_values.append(np.nan)
    else:
        gse_values.append(_M.get((int(o), e), np.nan))
 
# Crear variable categórica ordenada
df["gse"] = pd.Categorical(
    gse_values,
    categories=_ORDEN_GSE,
    ordered=True
)
 
# Resumen
print("="*80)
print("GSE DERIVADO — Distribución")
print("="*80)
print(df["gse"].value_counts().reindex(_ORDEN_GSE))
print(f"\nValores faltantes: {df['gse'].isna().sum()}")
print(f"Total casos: {len(df)}")
print(f"Cobertura: {(len(df) - df['gse'].isna().sum()) / len(df) * 100:.1f}%")
 
# Guardar
print("\nGuardando...")
df.to_csv('/tmp/EAUI2026_habilidades_socio.csv', index=False)
print("✓ Archivo actualizado con GSE")
 
 



GSE DERIVADO — Distribución
gse
AB     342
C1     533
C2     988
C3    1316
D      833
E      988
Name: count, dtype: int64

Valores faltantes: 0
Total casos: 5000
Cobertura: 100.0%

Guardando...
✓ Archivo actualizado con GSE


In [25]:
# ============================================================================
# FUNCIÓN DSTATS — Análisis Ponderado (AJUSTADA A NUEVAS VARIABLES)
# ============================================================================
# Uso: dstats(df, "variable", tipo="frecuencia", factor="fe_personas")
# Tipos: "frecuencia", "cruzada", "promedio", "suma"

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Diccionario de ordenamiento de categorías
# Actualiza según tus necesidades de presentación
ORDEN_CATEGORIAS = {
    "gse": ["AB", "C1", "C2", "C3", "D", "E"],
    "zona": ["URBANA", "RURAL"],
    "region": [
        "Arica", "Tarapacá", "Antofagasta", "Atacama", "Coquimbo", "Valparaíso",
        "Ohiggins", "Maule", "Biobío", "Araucanía", "Los Lagos", "Los Ríos",
        "Metropolitana", "Ñuble", "Aysen", "Magallanes"
    ],
    "sexo": ["Hombre", "Mujer"],
    "edad": ["15-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "educ": [
        "Sin educación formal", "Básica incompleta", "Básica completa",
        "Media Científico-Humanista Incompleta", "Media Científico-Humanista Completa",
        "Media Técnico-Profesional Incompleta", "Media Técnico-Profesional Completa",
        "Superior Técnica Incompleta", "Superior Técnica Completa",
        "Superior Universitaria Incompleta", "Superior Universitaria Completa"
    ],
    "ultimo_uso_internet": ["Hoy", "Ayer", "Hace 2-3 días", "Hace una semana o más", "No usa internet"],
    "frecuencia_internet": ["Varias veces al día", "Diaria", "4-5 días a la semana", "2-3 días a la semana", "1 día a la semana", "Rara vez", "Nunca"],
}

def _ordenar(df, variable):
    """Ordena DataFrame según ORDEN_CATEGORIAS si existe."""
    if variable not in ORDEN_CATEGORIAS:
        return df
    
    orden = ORDEN_CATEGORIAS[variable]
    idx_ordenado = [v for v in orden if v in df.index]
    idx_resto = [v for v in df.index if v not in idx_ordenado]
    
    return df.reindex(idx_ordenado + idx_resto)

def _mostrar(df, titulo):
    """Imprime tabla alineada y muestra DataFrame HTML."""
    print(f"\n{'='*80}")
    print(f"{titulo}")
    print(f"{'='*80}\n")
    print(df.to_string())
    print()
    display(df)

def dstats(data_df, variables, tipo="frecuencia", cruce=None, cruce2=None, factor=None,
           transponer=False, imprimir=True):
    """
    Análisis ponderado. Devuelve un DataFrame.

    Parámetros:
    -----------
    data_df : DataFrame
        Datos con variables y factor de expansión
    variables : str o list
        Variable(s) a analizar
    tipo : str
        "frecuencia" → recuento y %, "cruzada" → tabla pivote,
        "promedio" → promedio ponderado, "suma" → suma ponderada
    factor : str
        Factor de expansión ("fe_personas" o "fe_hogar")
    cruce : str
        Variable para cruzar (solo tipos "cruzada", "promedio", "suma")
    cruce2 : str
        Segunda variable de cruce (solo tipo "cruzada")
    transponer : bool
        Transpone tabla de cruces
    imprimir : bool
        Si True, imprime tabla y muestra HTML

    Retorna: DataFrame con análisis solicitado
    """
    if isinstance(variables, str):
        variables = [variables]
    
    # Validar columnas
    cols_requeridas = variables + ([factor] if factor else []) + ([cruce] if cruce else []) + ([cruce2] if cruce2 else [])
    for col in cols_requeridas:
        if col and col not in data_df.columns:
            raise ValueError(f"Columna '{col}' no existe.")

    # FRECUENCIA
    if tipo == "frecuencia":
        var = variables[0]
        tot = data_df[factor].sum()
        res = (data_df.groupby(var, observed=True)[factor].sum()
               .reset_index().rename(columns={factor: "n_ponderado"}))
        res["porcentaje"] = (res["n_ponderado"] / tot * 100).round(2)
        res = _ordenar(res, var).set_index(var)
        titulo = f"Frecuencia: '{var}' — base ponderada: {int(tot):,} ({factor})"
        if imprimir:
            _mostrar(res, titulo)
        return res

    # CRUZADA
    if tipo == "cruzada":
        var = variables[0]
        tot = data_df[factor].sum()

        # Cruce triple (var × cruce × cruce2)
        if cruce2:
            t = data_df.pivot_table(
                values=factor,
                index=var,
                columns=[cruce, cruce2],
                aggfunc="sum",
                fill_value=0,
                observed=True,
            )
            tp = t.div(t.sum(axis=0), axis=1).mul(100).round(2)

            # Ordenar índice
            if var in ORDEN_CATEGORIAS:
                of = [v for v in ORDEN_CATEGORIAS[var] if v in t.index]
                if of:
                    otros = [v for v in t.index if v not in of]
                    t, tp = t.reindex(of + otros), tp.reindex(of + otros)

            # Ordenar nivel 0 de columnas (cruce)
            if cruce in ORDEN_CATEGORIAS:
                oc = [v for v in ORDEN_CATEGORIAS[cruce] if v in t.columns.get_level_values(0)]
                if oc:
                    otros = [v for v in t.columns.get_level_values(0).unique() if v not in oc]
                    orden_cols = [(c, c2) for c in (oc + otros) for c2 in t.columns.get_level_values(1).unique() 
                                  if (c, c2) in t.columns]
                    t = t[orden_cols]
                    tp = tp[orden_cols]

            # Ordenar nivel 1 de columnas (cruce2)
            if cruce2 in ORDEN_CATEGORIAS:
                oc2 = [v for v in ORDEN_CATEGORIAS[cruce2] if v in t.columns.get_level_values(1)]
                if oc2:
                    columnas_ordenadas = [col for col in t.columns if col[1] in oc2]
                    columnas_otras = [col for col in t.columns if col[1] not in oc2]
                    t = t[columnas_ordenadas + columnas_otras]
                    tp = tp[columnas_ordenadas + columnas_otras]

            if transponer:
                t, tp = t.T, tp.T

            # Combinar n y % por columna
            cols = []
            for c1, c2 in t.columns:
                cols.append(t[(c1, c2)].rename(f"n {c1}|{c2}"))
                cols.append(tp[(c1, c2)].rename(f"% {c1}|{c2}"))
            
            res = pd.concat(cols, axis=1) if cols else pd.DataFrame()
            titulo = f"Cruce: '{var}' x '{cruce}' x '{cruce2}' — base: {int(tot):,} ({factor})"
            if imprimir:
                _mostrar(res, titulo)
            return res

        # Cruce simple (dos variables)
        t = data_df.pivot_table(
            values=factor,
            index=var,
            columns=cruce,
            aggfunc="sum",
            fill_value=0,
            observed=True,
        )
        tp = t.div(t.sum(axis=0), axis=1).mul(100).round(2)
        
        # Ordenar índice
        if var in ORDEN_CATEGORIAS:
            of = [v for v in ORDEN_CATEGORIAS[var] if v in t.index]
            if of:
                otros = [v for v in t.index if v not in of]
                t, tp = t.reindex(of + otros), tp.reindex(of + otros)
        
        # Ordenar columnas
        if cruce in ORDEN_CATEGORIAS:
            oc = [v for v in ORDEN_CATEGORIAS[cruce] if v in t.columns]
            if oc:
                otros = [v for v in t.columns if v not in oc]
                t, tp = t[oc + otros], tp[oc + otros]
        
        if transponer:
            t, tp = t.T, tp.T
        
        # Combinar n y %
        cols = []
        for c in t.columns:
            cols.append(t[c].rename(f"n {c}"))
            cols.append(tp[c].rename(f"% {c}"))
        
        res = pd.concat(cols, axis=1) if cols else pd.DataFrame()
        titulo = f"Cruce: '{var}' x '{cruce}' — base: {int(tot):,} ({factor})"
        if imprimir:
            _mostrar(res, titulo)
        return res

    # PROMEDIO y SUMA
    def _wavg(sub, v, f):
        """Promedio ponderado."""
        d = sub[[v, f]].dropna()
        if len(d) > 0 and pd.to_numeric(d[v], errors='coerce').notna().any():
            return float(round(np.average(pd.to_numeric(d[v], errors='coerce'), 
                                         weights=d[f], returned=False), 4))
        return np.nan

    def _wsum(sub, v, f):
        """Suma ponderada."""
        d = sub[[v, f]].dropna()
        if len(d) > 0:
            return float(round((pd.to_numeric(d[v], errors='coerce') * d[f]).sum(), 4))
        return np.nan

    fn = _wavg if tipo == "promedio" else _wsum
    col_name = "promedio_ponderado" if tipo == "promedio" else "suma_ponderada"

    # Sin cruce
    if not cruce:
        res = pd.DataFrame([(v, fn(data_df, v, factor)) for v in variables],
                           columns=["variable", col_name]).set_index("variable")
        titulo = f"{tipo.capitalize()} de variables — factor: {factor}"
        if imprimir:
            _mostrar(res, titulo)
        return res

    # Con cruce
    filas = {}
    for g, sg in data_df.groupby(cruce, observed=True):
        filas[g] = {v: fn(sg, v, factor) for v in variables}
    
    res = pd.DataFrame(filas).T
    res.index.name = cruce
    
    # Ordenar
    if cruce in ORDEN_CATEGORIAS:
        ok = [v for v in ORDEN_CATEGORIAS[cruce] if v in res.index]
        rst = [v for v in res.index if v not in ok]
        res = res.reindex(ok + rst)
    
    titulo = f"{tipo.capitalize()} cruzado por '{cruce}' — factor: {factor}"
    if imprimir:
        _mostrar(res, titulo)
    return res


# ============================================================================
# EJEMPLOS DE USO
# ============================================================================

# Cargar datos
# df = pd.read_csv('EAUI2026_habilidades_socio.csv')

# Frecuencia simple
# dstats(df, "gse", factor="fe_personas")
# dstats(df, "zona", factor="fe_personas")

# Cruce simple
# dstats(df, "gse", cruce="zona", factor="fe_personas", tipo="cruzada")
# dstats(df, "gse", cruce="sexo", factor="fe_personas", tipo="cruzada")

# Cruce triple
# dstats(df, "gse", cruce="zona", cruce2="sexo", factor="fe_personas", tipo="cruzada")

# Promedios cruzados (ej: edad promedio por GSE)
# dstats(df, "edad", cruce="gse", factor="fe_personas", tipo="promedio")

In [27]:
dstats(df, "gse", factor="fe_personas")
# dstats(df, "gse", cruce="zona", factor="fe_personas", tipo="cruzada")
# dstats(df, "gse", cruce="zona", cruce2="sexo", factor="fe_personas", tipo="cruzada")


Frecuencia: 'gse' — base ponderada: 13,810,761 (fe_personas)

      n_ponderado  porcentaje
gse                          
AB   1.522010e+06       11.02
C1   1.845934e+06       13.37
C2   3.203799e+06       23.20
C3   3.251371e+06       23.54
D    2.095975e+06       15.18
E    1.891672e+06       13.70



,n_ponderado,porcentaje
gse,,
AB,1.522010e+06,11.02
C1,1.845934e+06,13.37
C2,3.203799e+06,23.20
C3,3.251371e+06,23.54
D,2.095975e+06,15.18
E,1.891672e+06,13.70


,n_ponderado,porcentaje
gse,,
AB,1.522010e+06,11.02
C1,1.845934e+06,13.37
C2,3.203799e+06,23.20
C3,3.251371e+06,23.54
D,2.095975e+06,15.18
E,1.891672e+06,13.70
